Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr

Date: September 2024

In [2]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint
import time
import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

Define winter period and the date range that we are working with.

In [3]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'

## temporal window to smooth the data
smoothing_window_size = 3

## temporal window to deal with snow
required_snow_free_days=3
snow_free_window_size=5

## get the time period we care about
start_date_data = '2017-01-01'
end_date_data = '2023-12-31'


Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [4]:

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
# bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])

## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))


## just check on a couple of things
print('Projection of MODIS NDVI data:', modis_projection.getInfo())
print('Reprojected Bounding Box:', reprojected_bounding_box.getInfo())
print("Bands available in the first image (NDVI):", modis_ndvi.first().bandNames().getInfo())

print("Target projection:", target_projection)
print("Target scale:", target_scale)

print("Number of Images (ndvi):", modis_ndvi.size().getInfo())


Projection of MODIS NDVI data: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Reprojected Bounding Box: {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459407], [-9135043.795726, 4686382.625733], [-9195643.896612, 4639491.792059001]]]}
Bands available in the first image (NDVI): ['sur_refl_b01', 'sur_refl_b02', 'QA', 'State', 'NDVI']
Target projection: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Target scale: 231.6

Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

The second function creates a snow mask; this differs from the mask that Jerod and Shannon had made, this will instead create a mask if a NDVI pixel occurs at a point in time where 2 weeks prior and 1 week after the date of the NDVI image has average snow cover that exceeds a set threshold.


In [5]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    is_land = state.rightShift(3).bitwiseAnd(7).eq(1)  ## land (not in Jerod's code)
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice).And(is_land)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information from the State band of the MODIS data
def extract_snow(image):
    
    ## get the state band
    state = image.select('State')

    ## bit 12 represents snow/ice
    snow_mod35 = state.bitwiseAnd(1 << 12).neq(0)  # 1 if snow/ice is present, 0 otherwise

    ## bit 16 represents internal snow
    snow_internal = state.bitwiseAnd(1 << 16).neq(0)  # 1 if internal snow is present, 0 otherwise

    ## combine the MOD35 and internal snow
    snow_combined = snow_mod35.Or(snow_internal)  # 1 if either MOD35 or internal snow is present

    ## add the snow band to the image
    snow_band = snow_combined.rename('SnowCovered')

    ## carry over the system:time_start property from the original image (this is not saved in the State band, so you need to explicitly call it back in)
    ## and make sure it stays in the native projection

    snow_band = snow_band.set('system:time_start', image.get('system:time_start'))
    return snow_band


## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = modis_ndvi.map(extract_snow)

## just check on a couple of things
# print('Filtered MODIS information:', NDVIFiltered.getInfo())
print("Bands available in the first image:", NDVIFiltered.first().bandNames().getInfo())
print("Number of Images:", NDVIFiltered.size().getInfo())

# print('Filtered MODIS Snow information (NDVI):', SnowCover.getInfo())
print("Bands available in the first image (Snow):", SnowCover.first().bandNames().getInfo())
print("Number of Images:", SnowCover.size().getInfo())



Bands available in the first image: ['NDVI']
Number of Images: 322
Bands available in the first image (Snow): ['SnowCovered']
Number of Images: 322


Steps 1 - 4. Calculations. Reclassify outliers, reclassify winter values, check if a pixel has been snow-free for a set number of consecutive images, mask pizels that are in winter or covered by snow, apply median filter.

Step 1 in this section are the same as the steps in Merkle and Albeke's code for processing NDVI. Step 2 deviates from their methods. Merkle and Albeke methods require a pixel = to be snow free for 3 consecutive time steps in order to retain the NDVI value. This process doesn't work great in the more arid environments, so I shifted that methodology here. Instead, I required a minimum number of snow free days within a moving window. For now, I used a 5-image moving window with a requirement of at least 3 snow-free images within that window. This allows for brief snowfall events to be overlooked while still ensuring sufficient snow-free period in more temperate systems.

Step 3 is the same as the Merkle and Albeke methods.

In [6]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):
    
    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.select('NDVI').gte(0).And(image.select('NDVI').lte(2)))

    NDVICollection = NDVICollection.map(reclassify_outliers)
    return NDVICollection

## step 2: check if a pixel has been snow-free for at least N days within a moving window and
## update the snowcover collection
def ndviCALC_step2(image_collection, snow_free_window_size = 5, required_snow_free_days = 3):
    
    def calculate_snow_free_for_image(index):
        
        ## define the start and end of the N-image window
        start_index = ee.Number(index).subtract(snow_free_window_size - 1).max(0)
        end_index = ee.Number(index).add(1)

        ## get the moving window of images and select the 'Snow' band
        window_images = image_collection.toList(image_collection.size()).slice(start_index, end_index)
        window_collection = ee.ImageCollection.fromImages(window_images).select('SnowCovered')

        ## calculate the number of snow-free days (where Snow == 0) within the window
        snow_free_days = window_collection.map(lambda img: img.eq(0)).sum()

        ## get the current image
        current_image = ee.Image(image_collection.toList(image_collection.size()).get(index))

        ## create a binary mask where pixels are considered snow-covered if they do not meet the snow-free threshold
        snow_covered_mask = snow_free_days.lt(required_snow_free_days)  # Flip the threshold logic
        
        ## apply this binary mask to the current image
        result_image = ee.Algorithms.If(
            ee.Number(index).lt(snow_free_window_size - 1),
            current_image.select('SnowCovered'),  # use original image for early indices
            snow_covered_mask.rename('SnowCovered').set('system:time_start', current_image.get('system:time_start'))
        )

        ## this returns an image that has a binary 1/0 where 1 represents 'snow covered' pixels
        return result_image

    ## apply this function across each image in the collection
    indices = ee.List.sequence(0, image_collection.size().subtract(1))
    processed_collection = indices.map(lambda i: calculate_snow_free_for_image(i))
    
    return ee.ImageCollection.fromImages(processed_collection)
    

## step 3: apply median filter
def ndviCALC_step3(image_collection, window_size=smoothing_window_size):
    
    image_collection = image_collection.map(lambda img: img.select('NDVI').toFloat()) 

    def smooth_image(current_image):
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        image_index = all_images_list.indexOf(current_image)
        half_window = ee.Number(window_size // 2)

        ## calculate the start and end indices
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## get the window of images
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median
        median_image = window_collection.median().toFloat()  # Keep NDVI in float

        ## return the median image with the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    ## apply the smoothing function to the collection
    smoothed_collection = image_collection.map(smooth_image)
    return smoothed_collection

## run first 5 steps sequentially
NDVIProcessed1 = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")
print("Number of Images:", NDVIFiltered.size().getInfo())

SnowCover_MovingWindow = ndviCALC_step2(SnowCover, required_snow_free_days=3, snow_free_window_size=5)
print("Step 2 complete: Apply a mask based on the snow cover image collection.")
print("Number of Images:", NDVIFiltered.size().getInfo())

NDVIProcessed3 = ndviCALC_step3(NDVIProcessed1, window_size=smoothing_window_size)
print("Step 3 complete: Applied moving window with a window size of 3.")
print("Number of Images:", NDVIFiltered.size().getInfo())


Step 1 complete: Removed outliers from data.
Number of Images: 322
Step 2 complete: Apply a mask based on the snow cover image collection.
Number of Images: 322
Step 3 complete: Applied moving window with a window size of 3.
Number of Images: 322


Steps 5 - 8. Calculate the baseline value, mask NDVI values that are covered with snow, and floor values below the baseline and covered by snow during the winter period.

These steps also differ slightly from Jerod's methods. Step 4a and 4b generally acheives the same thing as Jerod and Shannon's methods, but I shifted steps 5 - 7. Here, it adds a band of yes/no winter and yes/no snow (based on both the snow mask image collectoin and the moving window snow mask image collection). Finally, it applied those masks and replaced any masked snow values with the baseline image value. You can decide if you want to floor all winter values as well.

In [7]:
## step 4: calculate the floor (0.025 quantile) without masking cells in the baseline image that have masked cells elsewhere in the image collection
## I was running into a lot of issues here trying to used the already masked image collections (NDVIProcessed4)
## so I am rerunning the masks. This works, I have no idea why using the existing image collections did not.

## first mask things
def ndviCALC_step4a(image):

    ## apply quality and snow masks
    quality_masked = quality_and_state_mask(image)
    snow_masked = SnowCover_MovingWindow.map(lambda snow_img: snow_img.select('SnowCovered').eq(0))  # Create a collection of snow-free masks
    snow_free_mask = snow_masked.max()  # Combine to get pixels that were snow-free at any point
    
    ## get rid of negative values
    range_masked = quality_masked.updateMask(image.select('NDVI').gte(0).And(image.select('NDVI').lte(2)))

    return range_masked.updateMask(snow_free_mask)

## then calcualte the baseline
def ndviCALC_step4b(masked_collection, quantile_value=2.5):
    quantile_fraction = quantile_value / 100.0
    # Calculate the 0.025 quantile across the entire time series for each pixel
    baseline_image = masked_collection.select('NDVI').reduce(
        ee.Reducer.percentile([quantile_fraction * 100])
    ).rename('NDVI_baseline')
    return baseline_image

## step 5: add a winter mask
def ndviCALC_step5(NDVICollection, start_month_day='11-01', end_month_day='03-30'):

    ## parse the start and end month-day strings
    start_month, start_day = map(int, start_month_day.split('-'))
    end_month, end_day = map(int, end_month_day.split('-'))
    
    def add_winter_mask(image):
        
        ## get the date and year of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for the current year
        winter_start = ee.Date.fromYMD(year, start_month, start_day)
        winter_end = ee.Date.fromYMD(year, end_month, end_day)
        
        ## create a condition to determine if the image is in the winter period
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(
            date.difference(winter_end, 'day').lte(0)
        )
        
        ## add the winter mask as a new band
        return image.addBands(ee.Image(is_in_winter).rename('winter_mask'))
    
    ## apply the winter mask to each image in the collection
    NDVICollection_with_winter_mask = NDVICollection.map(add_winter_mask)
    
    return NDVICollection_with_winter_mask

## step 6: add a snow mask layer
def ndviCALC_step6(NDVICollection, SnowCover_MovingWindow, SnowCover):

    def add_snow_masks(ndvi_image):
        
        ## get the NDVI date
        ndvi_date = ndvi_image.get('system:time_start')

        ## find the corresponding snow mask by matching dates
        moving_window_mask = SnowCover_MovingWindow.filter(ee.Filter.eq('system:time_start', ndvi_date)).first()
        original_snow_mask = SnowCover.filter(ee.Filter.eq('system:time_start', ndvi_date)).first()

        ## if there's no matching snow mask, use a default mask with no snow (0)
        moving_window_mask = ee.Image(ee.Algorithms.If(
            moving_window_mask, moving_window_mask.select('SnowCovered'), ee.Image.constant(0).rename('SnowCovered')
        ))
        
        original_snow_mask = ee.Image(ee.Algorithms.If(
            original_snow_mask, original_snow_mask.select('SnowCovered'), ee.Image.constant(0).rename('SnowCovered')
        ))
        
        combined_snow_mask = moving_window_mask.Or(original_snow_mask).rename('SnowCovered')

        ## apply the commbined band to the NDVI band
        ndvi_image_with_combined_mask = ndvi_image.addBands(combined_snow_mask)

        ## add the snow mask as a new band to the NDVI image
        return ndvi_image_with_combined_mask
    
    ## apply the snow mask addition to each image in the NDVI collection
    NDVICollection_with_masks = NDVICollection.map(add_snow_masks)
    
    return NDVICollection_with_masks

## step 7: apply corrections to the NDVI collection based on snow cover and winter periods
## you can turn on and off if you want it to replace all the winter values as the baseline
def ndviCALC_step7(NDVICollection, baseline_ndvi, restrict_to_winter=False):

    def apply_corrections(image):

        ## select masks and NDVI band
        winter_mask = image.select('winter_mask').reduce(ee.Reducer.max()).rename('winter_mask') ## ensure single band
        snow_mask = image.select('SnowCovered').reduce(ee.Reducer.max()).rename('SnowCovered') ## ensure single band
        ndvi = image.select('NDVI') ## NDVI values
        
        ## decide whether to restrict corrections to winter months
        if restrict_to_winter:
            correction_condition = winter_mask.And(snow_mask)
        else:
            correction_condition = snow_mask
        
        ## apply corrections, set NDVI to baseline where conditions are met
        corrected_ndvi = ndvi.where(correction_condition.Or(ndvi.lt(baseline_ndvi.select('NDVI_baseline'))), baseline_ndvi.select('NDVI_baseline'))
        corrected_ndvi = corrected_ndvi.unmask(baseline_ndvi.select('NDVI_baseline'))
        
        ## add the corrected NDVI as an updated band
        return image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)
    
    ## apply corrections to each image in the NDVI collection
    corrected_NDVICollection = NDVICollection.map(apply_corrections)
    
    return corrected_NDVICollection


# Step 4: Get the baseline image
step4a_tmp_imagecollection = modis_ndvi.map(ndviCALC_step4a) ## run the masks
baseline_ndvi_image = ndviCALC_step4b(step4a_tmp_imagecollection, quantile_value=2.5)
print("Step 4 commplete: Baseline NDVI Image (0.025 quantile) calculated successfully.")
# print("Baseline NDVI Image Info:", baseline_ndvi_image.getInfo())

# Step 5: Add the winter mask to each image
NDVIProcessed5 = ndviCALC_step5(NDVIProcessed3, start_month_day, end_month_day)
print("Step 5 complete: Add winter mask.")
print("Number of Images:", NDVIFiltered.size().getInfo())

# Step 6: Add the snow mask to each image
NDVIProcessed6 = ndviCALC_step6(NDVIProcessed5, SnowCover_MovingWindow, SnowCover)
print("Step 6 complete: Add snow mask.")
print("Number of Images:", NDVIFiltered.size().getInfo())

# Step 7: Apply corrections based on winter and snow mask
corrected_NDVI = ndviCALC_step7(NDVIProcessed6, baseline_ndvi_image, restrict_to_winter=False)
print("Step 7 complete: Add baseline values where snow exists.")
print("Number of Images:", NDVIFiltered.size().getInfo())
print("Bands available in final processed NDVI image collection:", corrected_NDVI.first().bandNames().getInfo())



Step 4 commplete: Baseline NDVI Image (0.025 quantile) calculated successfully.
Step 5 complete: Add winter mask.
Number of Images: 322
Step 6 complete: Add snow mask.
Number of Images: 322
Step 7 complete: Add baseline values where snow exists.
Number of Images: 322
Bands available in final processed NDVI image collection: ['NDVI', 'winter_mask', 'SnowCovered']


This next step is also new. Because we aren't just flooring across the entire winter, this next step adds another general smoothing function that will correct the pixel valuye to the baseline if the both the neighboring images are at the baseline.

In [8]:
# def apply_neighbor_baseline_correction(ndvi_image, baseline_image, all_images, target_crs, target_scale):
    
#     ## reproject the current NDVI image to the target CRS and scale
#     ndvi_image = ndvi_image.reproject(crs=target_crs, scale=target_scale)

#     ## reproject all images to the target CRS and scale
#     all_images_list = all_images.toList(all_images.size())
#     image_index = all_images_list.indexOf(ndvi_image)

#     previous_image = ee.Image(ee.Algorithms.If(image_index.gt(0), all_images_list.get(image_index.subtract(1)), None))
#     next_image = ee.Image(ee.Algorithms.If(image_index.lt(all_images.size().subtract(1)), all_images_list.get(image_index.add(1)), None))

#     ## only reproject if the image is not None
#     previous_image = ee.Algorithms.If(previous_image, ee.Image(previous_image).reproject(crs=target_crs, scale=target_scale), None)
#     next_image = ee.Algorithms.If(next_image, ee.Image(next_image).reproject(crs=target_crs, scale=target_scale), None)

#     def is_baseline(image):
#         return image.select('NDVI').eq(baseline_image.select('NDVI_baseline')).reduceRegion(
#             reducer=ee.Reducer.allNonZero(),
#             geometry=image.geometry(),
#             scale=target_scale
#         ).values().get(0)

#     prev_set_to_baseline = ee.Algorithms.If(previous_image, is_baseline(ee.Image(previous_image)), ee.Number(0))
#     next_set_to_baseline = ee.Algorithms.If(next_image, is_baseline(ee.Image(next_image)), ee.Number(0))

#     ## determine if both neighboring images are set to the baseline
#     both_neighbors_set_to_baseline = ee.Number(prev_set_to_baseline).And(ee.Number(next_set_to_baseline))

#     ## define corrections based on neighbors
#     def correct_to_baseline():
#         return baseline_image.select('NDVI_baseline').rename('NDVI')

#     def keep_current_ndvi():
#         return ndvi_image.select('NDVI')

#     ## apply corrections based on neighbors
#     corrected_ndvi = ee.Image(ee.Algorithms.If(both_neighbors_set_to_baseline, correct_to_baseline(), keep_current_ndvi()))
#     return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)


# ## apply_neighbor_check_to_collection function to run it for the entire image collection
# def apply_neighbor_check_to_collection(image_collection, baseline_image, target_crs, target_scale):
#     all_images = image_collection.sort('system:time_start')
    
#     def check_neighbors(image):
#         corrected_image = apply_neighbor_baseline_correction(image, baseline_image, all_images, target_crs, target_scale)
#         return corrected_image.select(['NDVI']).rename('NDVI')
#     corrected_collection = image_collection.map(check_neighbors)
#     return corrected_collection


# ## set the target projection and scale from your baseline NDVI or the first image in your collection
# target_projection = modis_ndvi.first().projection().getInfo()
# target_crs = target_projection['crs']
# target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

# ## apply the neighbor correction with consistent projections
# corrected_modis_ndvi = apply_neighbor_check_to_collection(corrected_NDVI, baseline_ndvi_image, target_crs, target_scale)
# print("Number of Images:", NDVIFiltered.size().getInfo())
# print("Bands available in final processed NDVI image collection:", corrected_modis_ndvi.first().bandNames().getInfo())



Number of Images: 322


KeyboardInterrupt: 

In [9]:
## remove all the image collections we no longer need
NDVIProcessed1 = None 
NDVIProcessed3 = None  
step4a_tmp_imagecollection = None  
NDVIProcessed5 = None  

In [10]:
# Import the geemap library for visualization
import geemap

## reduce the collection so we can plot it without running into memory issues
reduced_collection = corrected_NDVI.limit(20)

# Initialize the map centered on your area of interest
Map = geemap.Map(center=[42.98, -109.44], zoom=6)  # Adjust center coordinates as needed

first_snow_mask_image = SnowCover_MovingWindow.first()
first_snow_mask_image1 = SnowCover.first()
baseline_image = baseline_ndvi_image

## moving window sc
snowlist1 =  SnowCover_MovingWindow.toList(SnowCover_MovingWindow.size())
snow19 = ee.Image(snowlist1.get(19))  # Index 19 for the 20th image

## regular sc
snowlist2 =  SnowCover.toList(SnowCover.size())
snow19b = ee.Image(snowlist2.get(19))  # Index 19 for the 20th image

# NDVI collection
ndvi_list = reduced_collection.toList(reduced_collection.size())
first_ndvi_image = reduced_collection.first()
twentieth_ndvi_image = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

# Add the NDVI image to the map with a color palette for visualization
Map.addLayer(first_ndvi_image, 
             {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
             'Filtered NDVI (1)')

# Add the 20th NDVI image to the map
Map.addLayer(twentieth_ndvi_image, 
             {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
             'Filtered NDVI (20)')

Map.addLayer(twentieth_ndvi_image, 
             {'bands': 'SnowCovered', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Filtered NDVI - snow mask(20)')

# Add the snow mask to the map with a binary color palette
Map.addLayer(snow19, 
             {'bands': 'SnowCovered', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow Mask - Moving Window (image 20)')

Map.addLayer(snow19b, 
             {'bands': 'SnowCovered', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow Mask (image 20)')

# Add the baseline 
Map.addLayer(baseline_image, 
             {'bands': 'NDVI_baseline', 'min': 0, 'max': 1, 'palette': ['white', 'red']}, 
             'Baseline')

# Add a layer control panel
Map.addLayerControl()

# Display the map
Map


Map(center=[42.98, -109.44], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchData…

In [11]:
## Function to generate yearly date ranges
def generate_yearly_date_ranges(start_date, end_date):
    start_year = datetime.strptime(start_date, "%Y-%m-%d").year
    end_year = datetime.strptime(end_date, "%Y-%m-%d").year
    yearly_ranges = []

    for year in range(start_year, end_year + 1):
        start_of_year = f"{year}-01-01"
        end_of_year = f"{year}-12-31"
        yearly_ranges.append((start_of_year, end_of_year))

    return yearly_ranges

## Step 1: Generate Yearly Date Ranges
start_date_data = '2017-01-01'
end_date_data = '2018-12-31'
yearly_ranges = generate_yearly_date_ranges(start_date_data, end_date_data)

## Function to create a grid of bounding boxes
def create_grid(bounding_box, rows, cols):
    bounds = bounding_box.bounds().getInfo()['coordinates'][0]
    min_lon, min_lat = bounds[0]
    max_lon, max_lat = bounds[2]

    lon_step = (max_lon - min_lon) / cols
    lat_step = (max_lat - min_lat) / rows

    grid = []
    for i in range(rows):
        for j in range(cols):
            lon1 = min_lon + lon_step * j
            lat1 = min_lat + lat_step * i
            lon2 = lon1 + lon_step
            lat2 = lat1 + lat_step
            tile = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
            grid.append(tile)

    return grid

## Step 2: Create a grid of bounding boxes
rows, cols = 5, 5
grid = create_grid(bounding_box, rows, cols)

## Function to cast all bands to Float32
def cast_to_float32(image):
    bands = image.bandNames()
    casted_bands = bands.map(lambda b: image.select([b]).toFloat())
    return ee.Image.cat(casted_bands).set('system:time_start', image.get('system:time_start'))

## Step 3: Export each tile for each year
def export_yearly_tiles(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports'):
    for year_index, (start_date, end_date) in enumerate(yearly_ranges):
        yearly_NDVI = corrected_NDVI.filterDate(start_date, end_date)
        yearly_NDVI = yearly_NDVI.map(lambda image: image.select(['NDVI']).toFloat())  # Only keep NDVI band and cast to Float32
        print(f"Exporting data for year: {start_date} to {end_date}")

        for tile_index, tile in enumerate(grid):
            # Mosaic and clip the NDVI band for the current tile
            export_image = yearly_NDVI.mosaic().clip(tile)
            export_task = ee.batch.Export.image.toDrive(
                image=export_image,
                description=f'Wyoming_NDVI_{start_date[:4]}_Tile_{tile_index + 1}',
                scale=500,  # Adjust according to your resolution needs
                region=tile,
                fileFormat='GeoTIFF',
                folder=folder_name
            )
            export_task.start()
            print(f"Started export task for Year {start_date[:4]} Tile {tile_index + 1}")

## Perform the export
export_yearly_tiles(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports')


## Perform the export
export_yearly_tiles(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports')


Exporting data for year: 2017-01-01 to 2017-12-31
Started export task for Year 2017 Tile 1
Started export task for Year 2017 Tile 2
Started export task for Year 2017 Tile 3
Started export task for Year 2017 Tile 4
Started export task for Year 2017 Tile 5
Started export task for Year 2017 Tile 6
Started export task for Year 2017 Tile 7
Started export task for Year 2017 Tile 8
Started export task for Year 2017 Tile 9
Started export task for Year 2017 Tile 10
Started export task for Year 2017 Tile 11
Started export task for Year 2017 Tile 12
Started export task for Year 2017 Tile 13
Started export task for Year 2017 Tile 14
Started export task for Year 2017 Tile 15
Started export task for Year 2017 Tile 16
Started export task for Year 2017 Tile 17
Started export task for Year 2017 Tile 18
Started export task for Year 2017 Tile 19
Started export task for Year 2017 Tile 20
Started export task for Year 2017 Tile 21
Started export task for Year 2017 Tile 22
Started export task for Year 2017 T

Export the data (at each time step so we can plot and look at them)